In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>เวอร์ชันแพ็กเกจ</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ requirement ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
นอกจาก[การแสดงผล instruction บน circuit](/guides/visualize-circuits) แล้ว คุณอาจต้องการแสดงผลการจัดตาราง circuit โดยใช้เมธอด [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) ของ Qiskit การแสดงผลนี้ช่วยให้ระบุเวลาที่ Qubit ไม่ได้ทำงานได้อย่างรวดเร็ว อย่างไรก็ตาม เมธอดนี้ไม่คืนค่าที่ถูกต้องสำหรับ dynamic circuit ในการแสดงผลการจัดตาราง dynamic circuit ให้ใช้เมธอด `draw_circuit_schedule_timing` ตามที่อธิบายในส่วน [Qiskit Runtime support](#qr-support)

## ตัวอย่าง

ในการแสดงผลโปรแกรม circuit ที่จัดตารางแล้ว คุณสามารถเรียกใช้ฟังก์ชันนี้พร้อม control arguments ชุดหนึ่ง ลักษณะส่วนใหญ่ของรูปภาพผลลัพธ์สามารถปรับแต่งได้ผ่าน stylesheet แต่ไม่จำเป็นต้องทำ

### วาดด้วย stylesheet ค่าเริ่มต้น

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### วาดด้วย stylesheet สำหรับการดีบักโปรแกรม

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

คุณสามารถสร้างฟังก์ชัน generator หรือ layout แบบกำหนดเองและอัปเดต stylesheet ที่มีอยู่ด้วยฟังก์ชันเหล่านั้น วิธีนี้ช่วยให้คุณควบคุมลักษณะส่วนใหญ่ของรูปภาพผลลัพธ์ได้โดยไม่ต้องแก้ไข codebase ของ scheduled circuit drawer ดูตัวอย่างเพิ่มเติมได้ที่ API reference ของ [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer)
<span id="qr-support"></span>

## Qiskit Runtime support

แม้ว่า timeline drawer ที่ built-in ใน Qiskit จะมีประโยชน์สำหรับ static circuit แต่อาจไม่ได้แสดงผล timing ของ [dynamic circuit](/guides/classical-feedforward-and-control-flow) อย่างถูกต้อง เนื่องจาก implicit operation เช่น broadcasting และการกำหนด branch ในฐานะส่วนหนึ่งของการรองรับ dynamic circuit Qiskit Runtime จะคืนข้อมูล timing ของ circuit ที่ถูกต้องภายใน job result เมื่อมีการร้องขอ

> **Note:** - นี่คือฟังก์ชันทดลอง อยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลง
> - ฟังก์ชันนี้ใช้ได้เฉพาะกับ Qiskit Runtime Sampler job เท่านั้น
> - แม้ว่า circuit time รวมจะถูกคืนใน metadata ของ "compilation" แต่นี่ไม่ใช่เวลาที่ใช้สำหรับการเรียกเก็บเงิน (quantum time)

### เปิดใช้งานการดึงข้อมูล timing

ในการเปิดใช้งานการดึงข้อมูล timing ให้ตั้งค่า flag ทดลอง `scheduler_timing` เป็น `True` เมื่อรัน primitive job

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### เข้าถึงข้อมูล circuit timing
เมื่อร้องขอแล้ว ข้อมูล circuit timing สำหรับแต่ละ PUB จะถูกคืนใน metadata ของ job result ภายใต้ `["compilation"]["scheduler_timing"]["timing"]` field นี้มีข้อมูล timing แบบ raw ในการแสดงข้อมูล timing ให้ใช้เครื่องมือแสดงผลที่ built-in ตามที่อธิบายในส่วน [แสดงผล timing](#visualize-timings)

ใช้โค้ดต่อไปนี้เพื่อเข้าถึงข้อมูล circuit timing สำหรับ PUB แรก:

In [4]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### ทำความเข้าใจข้อมูล timing แบบ raw
แม้ว่าการแสดงผลข้อมูล circuit timing โดยใช้เมธอด `draw_circuit_schedule_timing` จะเป็นกรณีใช้งานที่พบบ่อยที่สุด แต่การทำความเข้าใจโครงสร้างของข้อมูล timing แบบ raw ที่คืนมาอาจเป็นประโยชน์ เช่น ช่วยให้ดึงข้อมูลเชิงโปรแกรมได้

ข้อมูล timing ที่คืนใน `["compilation"]["scheduler_timing"]["timing"]` คือรายการของ string แต่ละ string แทน instruction เดียวบน channel หนึ่ง และแบ่งด้วย comma เป็นประเภทข้อมูลต่อไปนี้:

- `Branch` - กำหนดว่า instruction อยู่ใน control flow (then / else) หรือ main branch
- `Instruction` - Gate และ Qubit ที่ต้องดำเนินการ
- `Channel` - channel ที่ถูกกำหนดด้วย instruction สามารถเป็นหนึ่งในต่อไปนี้:
      - `Qubit x` - drive channel สำหรับ qubit _x_
      - `AWGRx_y` (arbitrary waveform generator readout) - ใช้โดย readout channel เพื่อสื่อสารเมื่อวัด qubit อาร์กิวเมนต์ _x_ และ _y_ สอดคล้องกับ readout instrument ID และหมายเลข qubit ตามลำดับ
- `T0` - เวลาเริ่มต้น instruction ภายใน schedule ทั้งหมด
- `Duration` - ระยะเวลาของ instruction ในหน่วย _dt_ วินาที โดย 1 dt = 1 scheduling cycle คุณสามารถหาค่า `dt` ของ backend ได้โดยใช้ [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt)
- `Pulse` - ประเภทของ pulse operation ที่ใช้

ตัวอย่าง:

In [5]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### แสดงผล timing
ด้วย `qiskit-ibm-runtime` v0.43.0 ขึ้นไป คุณสามารถแสดงผล circuit timing ได้ ในการแสดงผล timing คุณต้องแปลง result metadata เป็น `fig` ก่อนโดยใช้[เมธอด `draw_circuit_schedule_timing`](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26) เมธอดนี้คืน figure ของ `plotly` ที่สามารถแสดงโดยตรง บันทึกลงไฟล์ หรือทั้งสองอย่าง ดูข้อมูลเพิ่มเติมเกี่ยวกับคำสั่ง `plotly` ที่ใช้ได้ที่ [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) และ [`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html)

In [6]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d5kk3cn853es738e01dg (DONE)


![Hovering over the output shows information such as the start, finish, and duration.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Example of a generated figure')
#### ทำความเข้าใจ figure ที่สร้างขึ้น
รูปภาพข้อมูล circuit timing ที่ผลิตโดย `draw_circuit_schedule_timing` สื่อข้อมูลต่อไปนี้:

- แกน X คือเวลาในหน่วย _dt_ วินาที โดย 1 dt = 1 scheduling cycle คุณสามารถหาค่า `dt` ของ backend ได้โดยใช้ [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt)
- แกน Y คือ channel (คิดว่า channel เป็นเครื่องมือที่ปล่อย pulse)
    - `Receive channel` - channel เดียวที่ไม่ใช่เครื่องมือในตัวเอง เป็น instruction ที่เล่นบนทุก channel ที่เป็นส่วนหนึ่งของขั้นตอนการสื่อสารกับ hub ในขณะนั้น
    - `Qubit x` - drive channel สำหรับ qubit x
    - `AWGRx_y` (arbitrary waveform generator readout) - ใช้โดย readout channel เพื่อสื่อสารเมื่อวัด qubit อาร์กิวเมนต์ _x_ และ _y_ สอดคล้องกับ readout instrument ID และหมายเลข qubit ตามลำดับ
    - `Hub` - ควบคุม broadcasting

นอกจากนี้ แต่ละ instruction มีรูปแบบ *X_Y* โดย *X* คือชื่อของ instruction และ *Y* คือประเภท pulse ประเภท `play` ใช้ control pulse และประเภท `capture` บันทึกสถานะของ qubit คุณยังสามารถวางเมาส์เหนือแต่ละ instruction เพื่อดูรายละเอียดเพิ่มเติม ตัวอย่างเช่น figure ก่อนหน้าแสดง control pulse สำหรับ X gate ที่ใช้กับ qubit 10 ที่ 1161 dt
### ตัวอย่างแบบ end-to-end
ตัวอย่างนี้แสดงวิธีเปิดใช้งาน option รับข้อมูล circuit timing จาก metadata และแสดงเป็นรูปภาพ

ขั้นแรก ตั้งค่าสภาพแวดล้อม กำหนด circuit และแปลงเป็น ISA circuit และกำหนดและรัน job

In [7]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,929,0,shift_phase\nmain,sx_0,Qubit 0,929,8,play\nmain,sx_0,Qubit 0,933,0,shift_phase\nmain,rz_0,Qubit 0,937,0,shift_phase\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 1,937,0,barrier\nmain,barrier,Qubit 2,937,0,barrier\nmain,barrier,Qubit 3,937,0,barrier\nmain,barrier,Qubit 4,937,0,barrier\nmain,barrier,Qubit 5,937,0,barrier\nmain,barrier,Qubit 6,937,0,barrier\nmain,barrier,Qubit 7,937,0,barrier\nmain,barrier,Qubit 8,937,0,barrier\nmain,barrier,Qubit 9,937,0,barrier\nmain,barrier,Qubit 10,937,0,barrier\nmain,barrier,Qubit 11,937,0,barrier\nmain,barrier,Qubit 12,937,0,barrier\nmain,barrier,Qubit 13,937,0,barrier\nmain,barrier,Qubit 14,937,0,barrier\nmain,barrier,Qubit 15,937,0,barrier\nmain,barrier,Qubit 16,937,0,barrier\nmain,barrier,Qubit 17,937,0,barrier\nmain,barrier,Qubit 18,937,0,barrier\nmain,barrier,Qubit 19,937,0,barrier\nmain,barrier,Qubit 20,937,0,barrier\nmain,barrier,Qubit 21,937,0,barrier\nmain,barrier,Qubit

Finally, you can visualize and save the timing:

In [8]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

ถัดไป รับ circuit schedule timing: